# QM9 structural panel — dataset construction
Circuit input graph: the QM9 bond graph (PyG `edge_index`, atom order = PyG order).
Chemistry (bond types, aromaticity, rings, stereo) comes from QM9's own SMILES via RDKit, renumbered to PyG order.
Every scalar target is an attribute on `Molecule`; `My_Dataset.columns` holds the stacked arrays.

In [ ]:
#!pip install torch_geometric rdkit   # uncomment when you run
import time
import pickle
import numpy as np
import networkx as nx
from networkx.algorithms import isomorphism
from rdkit import Chem, RDLogger
from torch_geometric.datasets import QM9
RDLogger.DisableLog("rdApp.*")        # 1819 QM9 records carry a five-bond carbon; the valence messages are handled below
from tqdm.notebook import tqdm

In [ ]:
QM9_ROOT = "qm9"
OUTPUT_PKL = "qm9_panel_columns.pkl"
PROGRESS_EVERY = 10000          # molecules between elapsed-time prints in the long loops
KEEP_WORKING_OBJECTS = False    # True keeps rdkit_mol, graph, heavy_graph and the matrices on every Molecule (~45 KB each, ~6 GB for all of QM9);
                                # False drops them after the panel is built; Molecule.rebuild_working_objects() brings them back on demand

ELEMENTS = np.array([1, 6, 7, 8, 9])                      # H C N O F, QM9's element set
BOND_TYPE_NAMES = {Chem.BondType.SINGLE: "single", Chem.BondType.DOUBLE: "double",
                   Chem.BondType.TRIPLE: "triple", Chem.BondType.AROMATIC: "aromatic"}

QM9_TARGET_NAMES = [
    "mu",            # D
    "alpha",         # a0^3
    "homo",          # eV
    "lumo",          # eV
    "gap",           # eV
    "r2",            # a0^2
    "zpve",          # eV
    "U0",            # eV
    "U",             # eV
    "H",             # eV
    "G",             # eV
    "Cv",            # cal / mol K
    "U0_atom",       # eV
    "U_atom",        # eV
    "H_atom",        # eV
    "G_atom",        # eV
    "A",             # GHz
    "B",             # GHz
    "C",             # GHz
]

# Section 3 functional motifs. Molecules carry explicit H atoms, so H counts in SMARTS are total H counts.
FUNCTIONAL_GROUP_SMARTS = {
    "num_hydroxyl":        "[#6;!$([#6]=O)]-[OX2H1]",
    "num_phenol":          "c-[OX2H1]",
    "num_ether":           "[#6]-[OX2;H0;!$(O-[#6]=O)]-[#6]",
    "num_carbonyl":        "[CX3]=[OX1]",
    "num_aldehyde":        "[CX3;H1,H2]=[OX1]",
    "num_ketone":          "[#6][CX3](=[OX1])[#6]",
    "num_carboxylic_acid": "[CX3](=O)[OX2H1]",
    "num_ester":           "[CX3](=O)[OX2;H0][#6]",
    "num_amide":           "[CX3](=O)[NX3]",
    "num_amine_primary":   "[NX3;H2;!$(N-C=O)][#6]",
    "num_amine_secondary": "[NX3;H1;!$(N-C=O)]([#6])[#6]",
    "num_amine_tertiary":  "[NX3;H0;!$(N-C=O)]([#6])([#6])[#6]",
    "num_nitrile":         "[NX1]#[CX2]",
    "num_nitro":           "[$([NX3](=O)=O),$([NX3+](=O)[O-])]",
    "num_alkene":          "[CX3]=[CX3]",
    "num_alkyne":          "[CX2]#[CX2]",
    "num_fluoro":          "[F]",
}
FUNCTIONAL_GROUP_PATTERNS = {name: Chem.MolFromSmarts(smarts) for name, smarts in FUNCTIONAL_GROUP_SMARTS.items()}
assert all(pattern is not None for pattern in FUNCTIONAL_GROUP_PATTERNS.values())

PANEL_SCALAR_FIELDS = [
    # section 1 composition
    "num_atoms", "num_heavy_atoms", "num_bonds_full", "num_bonds_heavy",
    "num_single_bonds_heavy", "num_double_bonds_heavy", "num_triple_bonds_heavy", "num_aromatic_bonds_heavy",
    # section 4.1 ring identity
    "num_rings", "num_aromatic_rings", "num_nonaromatic_rings",
    "num_3_member_rings", "num_4_member_rings", "num_5_member_rings", "num_6_member_rings", "num_7_plus_member_rings",
    "num_aromatic_5_member_rings", "num_aromatic_6_member_rings", "num_benzene_rings",
    "num_heteroaromatic_5_member_rings", "num_heteroaromatic_6_member_rings",
    # section 4.2 ring-system graph
    "num_benzene_isolated", "num_benzene_fused_terminal", "num_benzene_fused_interior", "num_benzene_fused_branch",
    "num_aromatic_isolated", "num_aromatic_fused_terminal", "num_aromatic_fused_interior", "num_aromatic_fused_branch",
    "num_aromatic_spiro",
    "num_ring_systems", "largest_ring_system_size", "num_fused_ring_systems", "num_spiro_centers", "max_ring_system_degree",
    # section 5 substitution context
    "num_aromatic_monosubstituted", "num_aromatic_disubstituted", "num_aromatic_trisubstituted_plus",
    "num_ortho_pairs", "num_meta_pairs", "num_para_pairs",
    # section 6 conjugation
    "num_conjugated_bonds", "num_conjugated_atoms", "num_conjugated_components", "largest_conjugated_component_atoms",
    "longest_conjugated_path", "num_aromatic_atoms", "aromatic_atom_fraction", "num_heteroatoms_in_conjugated_systems",
    "num_ring_conjugated_components",
    # section 7 generic invariants
    "num_bridges", "num_articulation_points", "cyclomatic_number", "girth",
    "diameter_heavy", "radius_heavy", "wiener_index_heavy", "average_shortest_path_length_heavy",
    "adjacency_spectral_radius_heavy", "laplacian_algebraic_connectivity_heavy",
    "diameter_full", "radius_full", "wiener_index_full", "adjacency_spectral_radius_full", "laplacian_algebraic_connectivity_full",
    "triangle_count", "cycle_4_count", "cycle_5_count", "cycle_6_count",
    # section 8 geometry
    "radius_of_gyration_full", "radius_of_gyration_heavy",
    "heavy_pairwise_distance_mean", "heavy_pairwise_distance_std", "heavy_pairwise_distance_min", "heavy_pairwise_distance_max",
    "bond_length_mean", "bond_length_std", "bond_length_min", "bond_length_max",
    "num_heavy_angles", "heavy_angle_mean", "heavy_angle_std", "heavy_angle_min", "heavy_angle_max",
    "num_heavy_torsions", "num_rotatable_bonds", "torsion_abs_cos_mean_rotatable", "torsion_abs_cos_mean_nonrotatable",
    # section 9 equivalence metadata
    "valence_error", "formula_class_id", "connectivity_class_id", "num_stereocenters", "num_unassigned_stereocenters",
    "num_duplicates", "num_constitutional_isomers", "num_enantiomers", "num_diastereomers",
    "has_constitutional_isomer", "has_enantiomer", "has_diastereomer",
] + list(FUNCTIONAL_GROUP_SMARTS.keys()) + QM9_TARGET_NAMES

PANEL_STRUCTURED_FIELDS = [
    "qm9_index", "name", "smiles", "canonical_smiles", "flat_smiles",
    "formula", "heavy_atom_formula", "element_counts", "bond_element_pair_counts",
    "element_degree_counts", "typed_radius1_environment_counts", "typed_radius2_environment_counts",
    "ring_records", "ring_pair_relations", "ring_size_counts",
    "aromatic_substitution_records",
    "conjugated_component_sizes",
    "degree_histogram_heavy", "degree_histogram_full",
    "heavy_pairwise_distance_spectrum", "bond_length_spectrum", "bond_lengths_by_type",
    "heavy_angle_spectrum", "heavy_angle_records", "heavy_torsion_records",
]

In [ ]:
dataset = QM9(root=QM9_ROOT)
print(dataset)
assert dataset[0].y.shape == (1, len(QM9_TARGET_NAMES)), dataset[0].y.shape
stereo_tagged = sum(("@" in dataset[i].smiles) or ("/" in dataset[i].smiles) or ("\\" in dataset[i].smiles) for i in range(len(dataset)))
print(f"molecules with stereo tags in smiles: {stereo_tagged} of {len(dataset)}")
assert stereo_tagged > 1000, "PyG smiles carry no stereo tags; stereo must be re-perceived from the SDF"

In [ ]:
def angle_between(position_a, position_b, position_c):
    """Angle in degrees at b formed by a-b-c"""
    vector_ba = position_a - position_b
    vector_bc = position_c - position_b
    cosine = np.dot(vector_ba, vector_bc) / (np.linalg.norm(vector_ba) * np.linalg.norm(vector_bc))
    return float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))


def dihedral_angle(position_a, position_b, position_c, position_d):
    """Signed dihedral in degrees for the chain a-b-c-d"""
    bond_1 = position_b - position_a
    bond_2 = position_c - position_b
    bond_3 = position_d - position_c
    normal_1 = np.cross(bond_1, bond_2)
    normal_2 = np.cross(bond_2, bond_3)
    unit_bond_2 = bond_2 / np.linalg.norm(bond_2)
    sine_term = np.dot(np.cross(normal_1, normal_2), unit_bond_2)
    cosine_term = np.dot(normal_1, normal_2)
    return float(np.degrees(np.arctan2(sine_term, cosine_term)))


def plane_rms_deviation(points):
    """RMS distance of points from their best-fit plane"""
    centered = points - points.mean(axis=0)
    _, singular_values, _ = np.linalg.svd(centered)
    return float(singular_values[-1] / np.sqrt(len(points)))


def longest_simple_path_length(graph):
    """Longest simple path (in edges) by exhaustive DFS; graphs here have at most 9 nodes"""
    best = 0
    def extend(node, visited, length):
        nonlocal best
        best = max(best, length)
        for neighbor in graph.neighbors(node):
            if neighbor not in visited:
                extend(neighbor, visited | {neighbor}, length + 1)
    for start in graph.nodes:
        extend(start, {start}, 0)
    return best

In [ ]:
class Molecule:
    def __init__(self, data, idx):
        self.idx = idx                              # position in My_Dataset.molecules
        self.qm9_index = int(data.idx)              # position in the PyG dataset
        self.name = data.name
        self.smiles = data.smiles
        self.atom_node_list = data.z.numpy().astype(int)
        self.positions = data.pos.numpy().astype(float)
        self.targets = data.y.numpy().astype(float)[0]
        self.num_atoms = len(self.atom_node_list)
        self.formula = tuple(int(count) for count in (self.atom_node_list[:, None] == ELEMENTS).sum(axis=0))
        self.heavy_atom_indices = [int(index) for index in np.where(self.atom_node_list > 1)[0]]

        edge_index = data.edge_index.numpy()
        self.pyg_edges = sorted({(int(min(a, b)), int(max(a, b))) for a, b in edge_index.T})

        self.rebuild_working_objects()

        heavy_mol = Chem.RemoveHs(self.rdkit_mol, sanitize=False)
        self.canonical_smiles = Chem.MolToSmiles(heavy_mol)
        self.flat_smiles = Chem.MolToSmiles(heavy_mol, isomericSmiles=False)
        self.mirror_smiles = self.build_mirror_smiles(heavy_mol)
        stereocenters = Chem.FindMolChiralCenters(heavy_mol, includeUnassigned=True, useLegacyImplementation=False)
        self.num_stereocenters = len(stereocenters)
        self.num_unassigned_stereocenters = sum(label == "?" for _, label in stereocenters)

        self.formula_class_id = None
        self.connectivity_class_id = None
        self.duplicates = []
        self.constitutional_isomers = []
        self.enantiomers = []
        self.diastereomers = []
        self.stereo_undefined_pairs = []

        self.build_panel_structure()
        if not KEEP_WORKING_OBJECTS:
            self.release_working_objects()

    def rebuild_working_objects(self):
        """RDKit mol in PyG order, the bond graph, its heavy-atom subgraph, and the two matrices"""
        self.rdkit_mol = self.build_aligned_rdkit_mol()
        self.graph = self.build_graph()
        self.heavy_graph = self.graph.subgraph(self.heavy_atom_indices).copy()
        self.adjacency_matrix = nx.to_numpy_array(self.graph, nodelist=range(self.num_atoms), dtype=int)
        self.bond_length_matrix = nx.to_numpy_array(self.graph, nodelist=range(self.num_atoms), weight="length")

    def release_working_objects(self):
        """Drops the objects rebuild_working_objects makes; every panel attribute stays"""
        del self.rdkit_mol, self.graph, self.heavy_graph, self.adjacency_matrix, self.bond_length_matrix

    # ---------------- construction ----------------
    def build_aligned_rdkit_mol(self):
        """Parses QM9's smiles (explicit H, stereo tags) and renumbers atoms into PyG order"""
        parser_params = Chem.SmilesParserParams()
        parser_params.removeHs = False
        mol = Chem.MolFromSmiles(self.smiles, parser_params)
        self.valence_error = mol is None
        if mol is None:
            # QM9's own bond table gives some carbons five bonds (1819 records). Keep the record as QM9 wrote it and
            # skip only RDKit's valence check; aromaticity, rings, conjugation, and stereo are still perceived.
            parser_params.sanitize = False
            mol = Chem.MolFromSmiles(self.smiles, parser_params)
            assert mol is not None, f"{self.name}: RDKit could not parse {self.smiles}"
            mol.UpdatePropertyCache(strict=False)
            sanitize_result = Chem.SanitizeMol(mol, sanitizeOps=Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_PROPERTIES, catchErrors=True)
            assert sanitize_result == Chem.SanitizeFlags.SANITIZE_NONE, f"{self.name}: partial sanitization failed with {sanitize_result}"
        assert mol.GetNumAtoms() == self.num_atoms, f"{self.name}: smiles has {mol.GetNumAtoms()} atoms, PyG has {self.num_atoms}"

        pyg_graph = nx.Graph()
        pyg_graph.add_nodes_from((index, {"element": int(z)}) for index, z in enumerate(self.atom_node_list))
        pyg_graph.add_edges_from(self.pyg_edges)
        rdkit_graph = nx.Graph()
        rdkit_graph.add_nodes_from((atom.GetIdx(), {"element": atom.GetAtomicNum()}) for atom in mol.GetAtoms())
        rdkit_graph.add_edges_from((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()) for bond in mol.GetBonds())
        matcher = isomorphism.GraphMatcher(pyg_graph, rdkit_graph, node_match=isomorphism.categorical_node_match("element", None))
        assert matcher.is_isomorphic(), f"{self.name}: PyG bond graph and smiles graph are not isomorphic"
        pyg_to_rdkit = matcher.mapping
        new_order = [pyg_to_rdkit[pyg_index] for pyg_index in range(self.num_atoms)]
        mol = Chem.RenumberAtoms(mol, new_order)

        rdkit_edges = sorted({(min(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()), max(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())) for bond in mol.GetBonds()})
        assert rdkit_edges == self.pyg_edges, f"{self.name}: bond lists disagree after renumbering"
        assert [atom.GetAtomicNum() for atom in mol.GetAtoms()] == list(self.atom_node_list), f"{self.name}: element order disagrees after renumbering"
        return mol

    def build_graph(self):
        """NetworkX graph in PyG order: element on nodes, RDKit bond type and geometric length on edges"""
        graph = nx.Graph()
        graph.add_nodes_from((index, {"element": int(z)}) for index, z in enumerate(self.atom_node_list))
        for bond in self.rdkit_mol.GetBonds():
            atom_a, atom_b = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            length = float(np.linalg.norm(self.positions[atom_a] - self.positions[atom_b]))
            graph.add_edge(atom_a, atom_b, bond_type=BOND_TYPE_NAMES[bond.GetBondType()], length=length,
                           conjugated=bond.GetIsConjugated(), aromatic=bond.GetIsAromatic(), in_ring=bond.IsInRing())
        return graph

    def build_mirror_smiles(self, heavy_mol):
        """Canonical smiles of the mirror image: every tetrahedral tag inverted, double-bond stereo untouched"""
        mirror = Chem.RWMol(heavy_mol)
        for atom in mirror.GetAtoms():
            if atom.GetChiralTag() == Chem.ChiralType.CHI_TETRAHEDRAL_CW:
                atom.SetChiralTag(Chem.ChiralType.CHI_TETRAHEDRAL_CCW)
            elif atom.GetChiralTag() == Chem.ChiralType.CHI_TETRAHEDRAL_CCW:
                atom.SetChiralTag(Chem.ChiralType.CHI_TETRAHEDRAL_CW)
        return Chem.MolToSmiles(mirror.GetMol())

    # ---------------- section 1: composition ----------------
    def build_panel_composition(self):
        self.num_heavy_atoms = len(self.heavy_atom_indices)
        self.num_bonds_full = self.graph.number_of_edges()
        self.num_bonds_heavy = self.heavy_graph.number_of_edges()
        self.element_counts = {int(element): int(count) for element, count in zip(ELEMENTS, self.formula)}
        self.heavy_atom_formula = tuple(count for element, count in zip(ELEMENTS, self.formula) if element > 1)

        heavy_bond_types = [attributes["bond_type"] for _, _, attributes in self.heavy_graph.edges(data=True)]
        self.num_single_bonds_heavy = heavy_bond_types.count("single")
        self.num_double_bonds_heavy = heavy_bond_types.count("double")
        self.num_triple_bonds_heavy = heavy_bond_types.count("triple")
        self.num_aromatic_bonds_heavy = heavy_bond_types.count("aromatic")

        bond_element_pair_counts = {}
        for atom_a, atom_b, attributes in self.graph.edges(data=True):
            key = (min(int(self.atom_node_list[atom_a]), int(self.atom_node_list[atom_b])),
                   max(int(self.atom_node_list[atom_a]), int(self.atom_node_list[atom_b])),
                   attributes["bond_type"])
            bond_element_pair_counts[key] = bond_element_pair_counts.get(key, 0) + 1
        self.bond_element_pair_counts = bond_element_pair_counts

    # ---------------- section 2: typed local structure ----------------
    def build_panel_environments(self):
        """Radius-1 and radius-2 typed environments of every heavy atom; H neighbors included"""
        radius1 = {}
        for atom in self.heavy_atom_indices:
            neighbor_terms = sorted((int(self.atom_node_list[neighbor]), self.graph.edges[atom, neighbor]["bond_type"]) for neighbor in self.graph.neighbors(atom))
            radius1[atom] = (int(self.atom_node_list[atom]), self.graph.degree[atom], tuple(neighbor_terms))
        radius1_or_hydrogen = dict(radius1)
        for atom in range(self.num_atoms):
            if atom not in radius1:
                heavy_neighbor = next(iter(self.graph.neighbors(atom)))
                radius1_or_hydrogen[atom] = (1, 1, ((int(self.atom_node_list[heavy_neighbor]), "single"),))

        self.element_degree_counts = {}
        self.typed_radius1_environment_counts = {}
        self.typed_radius2_environment_counts = {}
        for atom in self.heavy_atom_indices:
            element_degree = (int(self.atom_node_list[atom]), self.graph.degree[atom])
            self.element_degree_counts[element_degree] = self.element_degree_counts.get(element_degree, 0) + 1
            key1 = str(radius1[atom])
            self.typed_radius1_environment_counts[key1] = self.typed_radius1_environment_counts.get(key1, 0) + 1
            neighbor_environments = tuple(sorted(str(radius1_or_hydrogen[neighbor]) for neighbor in self.graph.neighbors(atom)))
            key2 = str((radius1[atom], neighbor_environments))
            self.typed_radius2_environment_counts[key2] = self.typed_radius2_environment_counts.get(key2, 0) + 1

    # ---------------- section 4.1: ring identity ----------------
    def build_panel_rings(self):
        ring_info = self.rdkit_mol.GetRingInfo()
        self.ring_records = []
        for ring_atoms, ring_bonds in zip(ring_info.AtomRings(), ring_info.BondRings()):
            bonds = [self.rdkit_mol.GetBondWithIdx(bond_index) for bond_index in ring_bonds]
            elements = tuple(int(self.atom_node_list[atom]) for atom in ring_atoms)
            external_attachments = 0
            for atom in ring_atoms:
                external_attachments += sum(neighbor not in ring_atoms and self.atom_node_list[neighbor] > 1 for neighbor in self.graph.neighbors(atom))
            ring_positions = self.positions[list(ring_atoms)]
            ring_lengths = [self.graph.edges[bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()]["length"] for bond in bonds]
            ring_angles = []
            size = len(ring_atoms)
            for position in range(size):
                previous_atom, atom, next_atom = ring_atoms[position - 1], ring_atoms[position], ring_atoms[(position + 1) % size]
                ring_angles.append(angle_between(self.positions[previous_atom], self.positions[atom], self.positions[next_atom]))
            self.ring_records.append({
                "atoms": tuple(int(atom) for atom in ring_atoms),
                "bond_indices": tuple(int(bond_index) for bond_index in ring_bonds),
                "size": size,
                "aromatic": all(bond.GetIsAromatic() for bond in bonds),
                "element_sequence": elements,
                "num_heteroatoms": sum(element not in (1, 6) for element in elements),
                "num_aromatic_bonds": sum(bond.GetIsAromatic() for bond in bonds),
                "num_external_attachments": external_attachments,
                "is_benzene": size == 6 and all(bond.GetIsAromatic() for bond in bonds) and all(element == 6 for element in elements),
                "bond_length_std": float(np.std(ring_lengths)),
                "angle_std": float(np.std(ring_angles)),
                "planarity_rms": plane_rms_deviation(ring_positions),
            })

        records = self.ring_records
        self.num_rings = len(records)
        self.num_aromatic_rings = sum(record["aromatic"] for record in records)
        self.num_nonaromatic_rings = self.num_rings - self.num_aromatic_rings
        self.ring_size_counts = {}
        for record in records:
            self.ring_size_counts[record["size"]] = self.ring_size_counts.get(record["size"], 0) + 1
        self.num_3_member_rings = self.ring_size_counts.get(3, 0)
        self.num_4_member_rings = self.ring_size_counts.get(4, 0)
        self.num_5_member_rings = self.ring_size_counts.get(5, 0)
        self.num_6_member_rings = self.ring_size_counts.get(6, 0)
        self.num_7_plus_member_rings = sum(count for size, count in self.ring_size_counts.items() if size >= 7)
        self.num_aromatic_5_member_rings = sum(record["aromatic"] and record["size"] == 5 for record in records)
        self.num_aromatic_6_member_rings = sum(record["aromatic"] and record["size"] == 6 for record in records)
        self.num_benzene_rings = sum(record["is_benzene"] for record in records)
        self.num_heteroaromatic_5_member_rings = sum(record["aromatic"] and record["size"] == 5 and record["num_heteroatoms"] > 0 for record in records)
        self.num_heteroaromatic_6_member_rings = sum(record["aromatic"] and record["size"] == 6 and record["num_heteroatoms"] > 0 for record in records)

    # ---------------- section 4.2: ring-system graph ----------------
    def build_panel_ring_systems(self):
        records = self.ring_records
        ring_graph = nx.Graph()
        ring_graph.add_nodes_from(range(len(records)))
        self.ring_pair_relations = []
        for ring_a in range(len(records)):
            for ring_b in range(ring_a + 1, len(records)):
                shared_atoms = set(records[ring_a]["atoms"]) & set(records[ring_b]["atoms"])
                shared_bonds = set(records[ring_a]["bond_indices"]) & set(records[ring_b]["bond_indices"])
                if len(shared_bonds) > 0:
                    relation = "fused"
                elif len(shared_atoms) == 1:
                    relation = "spiro"
                elif len(shared_atoms) > 1:
                    relation = "bridged"          # atoms shared without a shared bond
                else:
                    relation = "none"
                if relation != "none":
                    ring_graph.add_edge(ring_a, ring_b, relation=relation, shared_atoms=shared_atoms, shared_bonds=shared_bonds)
                    self.ring_pair_relations.append((ring_a, ring_b, relation, len(shared_atoms), len(shared_bonds)))

        spiro_atoms = set()
        for record_index, record in enumerate(records):
            shared_atoms = set()
            shared_bonds = set()
            fusion_junction_atoms = set()
            for _, _, attributes in ring_graph.edges(record_index, data=True):
                shared_atoms |= attributes["shared_atoms"]
                shared_bonds |= attributes["shared_bonds"]
                if attributes["relation"] == "fused":
                    fusion_junction_atoms |= attributes["shared_atoms"]
                if attributes["relation"] == "spiro":
                    spiro_atoms |= attributes["shared_atoms"]
            degree = ring_graph.degree[record_index]
            record["ring_system_degree"] = degree
            record["num_shared_atoms"] = len(shared_atoms)
            record["num_shared_bonds"] = len(shared_bonds)
            record["num_fusion_junction_atoms"] = len(fusion_junction_atoms)
            record["is_spiro"] = any(attributes["relation"] == "spiro" for _, _, attributes in ring_graph.edges(record_index, data=True))
            record["role"] = {0: "isolated", 1: "terminal", 2: "interior"}.get(degree, "branch")

        def count_roles(selector):
            return {role: sum(record["role"] == role for record in records if selector(record)) for role in ("isolated", "terminal", "interior", "branch")}
        benzene_roles = count_roles(lambda record: record["is_benzene"])
        aromatic_roles = count_roles(lambda record: record["aromatic"])
        self.num_benzene_isolated = benzene_roles["isolated"]
        self.num_benzene_fused_terminal = benzene_roles["terminal"]
        self.num_benzene_fused_interior = benzene_roles["interior"]
        self.num_benzene_fused_branch = benzene_roles["branch"]
        self.num_aromatic_isolated = aromatic_roles["isolated"]
        self.num_aromatic_fused_terminal = aromatic_roles["terminal"]
        self.num_aromatic_fused_interior = aromatic_roles["interior"]
        self.num_aromatic_fused_branch = aromatic_roles["branch"]
        self.num_aromatic_spiro = sum(record["aromatic"] and record["is_spiro"] for record in records)

        components = list(nx.connected_components(ring_graph))
        self.num_ring_systems = len(components)
        self.largest_ring_system_size = max((len(component) for component in components), default=0)
        fused_only = ring_graph.edge_subgraph([edge for edge in ring_graph.edges if ring_graph.edges[edge]["relation"] == "fused"]) if ring_graph.number_of_edges() > 0 else nx.Graph()
        self.num_fused_ring_systems = sum(len(component) >= 2 for component in nx.connected_components(fused_only)) if fused_only.number_of_nodes() > 0 else 0
        self.num_spiro_centers = len(spiro_atoms)
        self.max_ring_system_degree = max((ring_graph.degree[node] for node in ring_graph.nodes), default=0)

    # ---------------- section 5: substitution context ----------------
    def build_panel_substitution(self):
        """External heavy substituents on aromatic rings; atoms of a fused partner ring are not substituents"""
        records = self.ring_records
        self.aromatic_substitution_records = []
        self.num_aromatic_monosubstituted = 0
        self.num_aromatic_disubstituted = 0
        self.num_aromatic_trisubstituted_plus = 0
        self.num_ortho_pairs = 0
        self.num_meta_pairs = 0
        self.num_para_pairs = 0
        for record_index, record in enumerate(records):
            if not record["aromatic"]:
                continue
            ring_atoms = record["atoms"]
            fused_partner_atoms = set()
            for ring_a, ring_b, relation, _, _ in self.ring_pair_relations:
                if relation == "fused" and record_index in (ring_a, ring_b):
                    partner = ring_b if ring_a == record_index else ring_a
                    fused_partner_atoms |= set(records[partner]["atoms"])
            substituents = []
            for position, atom in enumerate(ring_atoms):
                for neighbor in self.graph.neighbors(atom):
                    if neighbor in ring_atoms or neighbor in fused_partner_atoms or self.atom_node_list[neighbor] == 1:
                        continue
                    attachment_bond = self.graph.edges[atom, neighbor]
                    other_bonds = [self.graph.edges[neighbor, second] for second in self.graph.neighbors(neighbor) if second != atom]
                    begins_conjugated_system = any(bond["conjugated"] for bond in other_bonds)
                    radius1_atoms = [second for second in self.graph.neighbors(neighbor) if second != atom]
                    radius2_atoms = [third for second in radius1_atoms for third in self.graph.neighbors(second) if third not in (neighbor, atom)]
                    heteroatom_radius1 = self.atom_node_list[neighbor] not in (1, 6) or any(self.atom_node_list[second] not in (1, 6) for second in radius1_atoms)
                    heteroatom_radius2 = heteroatom_radius1 or any(self.atom_node_list[third] not in (1, 6) for third in radius2_atoms)
                    substituents.append({
                        "ring_position": position,
                        "ring_atom": int(atom),
                        "attached_atom": int(neighbor),
                        "attached_element": int(self.atom_node_list[neighbor]),
                        "bond_type": attachment_bond["bond_type"],
                        "begins_conjugated_system": bool(begins_conjugated_system),
                        "heteroatom_within_radius1": bool(heteroatom_radius1),
                        "heteroatom_within_radius2": bool(heteroatom_radius2),
                    })
            separations = []
            if record["size"] == 6:
                for first in range(len(substituents)):
                    for second in range(first + 1, len(substituents)):
                        gap = abs(substituents[first]["ring_position"] - substituents[second]["ring_position"])
                        separation = min(gap, 6 - gap)
                        separations.append(separation)
                        if separation == 1:
                            self.num_ortho_pairs += 1
                        elif separation == 2:
                            self.num_meta_pairs += 1
                        elif separation == 3:
                            self.num_para_pairs += 1
            if len(substituents) == 1:
                self.num_aromatic_monosubstituted += 1
            elif len(substituents) == 2:
                self.num_aromatic_disubstituted += 1
            elif len(substituents) >= 3:
                self.num_aromatic_trisubstituted_plus += 1
            record["num_external_substituents"] = len(substituents)
            self.aromatic_substitution_records.append({
                "ring_index": record_index,
                "ring_size": record["size"],
                "substituents": substituents,
                "cyclic_separations": sorted(separations),
                "signature": tuple(sorted((s["attached_element"], s["bond_type"], s["begins_conjugated_system"], s["heteroatom_within_radius2"]) for s in substituents)),
            })

    # ---------------- section 3: functional motifs ----------------
    def build_panel_functional_groups(self):
        for name, pattern in FUNCTIONAL_GROUP_PATTERNS.items():
            setattr(self, name, len(self.rdkit_mol.GetSubstructMatches(pattern)))

    # ---------------- section 6: conjugation ----------------
    def build_panel_conjugation(self):
        conjugated_edges = [(a, b) for a, b, attributes in self.graph.edges(data=True) if attributes["conjugated"]]
        conjugated_graph = nx.Graph()
        conjugated_graph.add_edges_from(conjugated_edges)
        components = [conjugated_graph.subgraph(component).copy() for component in nx.connected_components(conjugated_graph)]
        self.num_conjugated_bonds = len(conjugated_edges)
        self.num_conjugated_atoms = conjugated_graph.number_of_nodes()
        self.num_conjugated_components = len(components)
        self.conjugated_component_sizes = sorted((component.number_of_nodes() for component in components), reverse=True)
        self.largest_conjugated_component_atoms = self.conjugated_component_sizes[0] if components else 0
        self.longest_conjugated_path = max((longest_simple_path_length(component) for component in components), default=0)
        aromatic_atoms = [atom.GetIdx() for atom in self.rdkit_mol.GetAtoms() if atom.GetIsAromatic()]
        self.num_aromatic_atoms = len(aromatic_atoms)
        self.aromatic_atom_fraction = len(aromatic_atoms) / len(self.heavy_atom_indices)
        self.num_heteroatoms_in_conjugated_systems = sum(self.atom_node_list[atom] not in (1, 6) for atom in conjugated_graph.nodes)
        self.num_ring_conjugated_components = sum(any(self.graph.edges[a, b]["in_ring"] for a, b in component.edges) for component in components)

    # ---------------- section 7: generic graph invariants ----------------
    def build_panel_graph_invariants(self):
        heavy = self.heavy_graph
        full = self.graph
        assert nx.is_connected(full), f"{self.name}: disconnected bond graph"
        heavy_degrees = [degree for _, degree in heavy.degree()]
        full_degrees = [degree for _, degree in full.degree()]
        self.degree_histogram_heavy = tuple(heavy_degrees.count(degree) for degree in range(6))   # degree 5 occurs in the valence_error records
        self.degree_histogram_full = tuple(full_degrees.count(degree) for degree in range(6))
        self.num_bridges = len(list(nx.bridges(heavy)))
        self.num_articulation_points = len(list(nx.articulation_points(heavy)))
        self.cyclomatic_number = heavy.number_of_edges() - heavy.number_of_nodes() + 1
        cycle_lengths = [len(cycle) for cycle in nx.simple_cycles(heavy)]
        self.girth = min(cycle_lengths) if cycle_lengths else 0      # 0 means acyclic
        self.triangle_count = cycle_lengths.count(3)
        self.cycle_4_count = cycle_lengths.count(4)
        self.cycle_5_count = cycle_lengths.count(5)
        self.cycle_6_count = cycle_lengths.count(6)

        for graph, suffix in ((heavy, "heavy"), (full, "full")):
            n_nodes = graph.number_of_nodes()
            if n_nodes == 1:
                setattr(self, f"diameter_{suffix}", 0)
                setattr(self, f"radius_{suffix}", 0)
                setattr(self, f"wiener_index_{suffix}", 0)
                setattr(self, f"adjacency_spectral_radius_{suffix}", 0.0)
                setattr(self, f"laplacian_algebraic_connectivity_{suffix}", 0.0)
                continue
            eccentricities = nx.eccentricity(graph)
            setattr(self, f"diameter_{suffix}", max(eccentricities.values()))
            setattr(self, f"radius_{suffix}", min(eccentricities.values()))
            setattr(self, f"wiener_index_{suffix}", int(nx.wiener_index(graph)))
            adjacency = nx.to_numpy_array(graph)
            setattr(self, f"adjacency_spectral_radius_{suffix}", float(np.max(np.linalg.eigvalsh(adjacency))))
            laplacian_eigenvalues = np.sort(np.linalg.eigvalsh(np.diag(adjacency.sum(axis=1)) - adjacency))
            setattr(self, f"laplacian_algebraic_connectivity_{suffix}", float(laplacian_eigenvalues[1]))
        self.average_shortest_path_length_heavy = float(nx.average_shortest_path_length(heavy)) if heavy.number_of_nodes() > 1 else 0.0

    # ---------------- section 8: geometry ----------------
    def build_panel_geometry(self):
        heavy_positions = self.positions[self.heavy_atom_indices]
        self.radius_of_gyration_full = float(np.sqrt(((self.positions - self.positions.mean(axis=0)) ** 2).sum(axis=1).mean()))
        self.radius_of_gyration_heavy = float(np.sqrt(((heavy_positions - heavy_positions.mean(axis=0)) ** 2).sum(axis=1).mean()))

        heavy_pairwise = [float(np.linalg.norm(heavy_positions[a] - heavy_positions[b])) for a in range(len(heavy_positions)) for b in range(a + 1, len(heavy_positions))]
        self.heavy_pairwise_distance_spectrum = sorted(heavy_pairwise)
        self.heavy_pairwise_distance_mean = float(np.mean(heavy_pairwise)) if heavy_pairwise else np.nan
        self.heavy_pairwise_distance_std = float(np.std(heavy_pairwise)) if heavy_pairwise else np.nan
        self.heavy_pairwise_distance_min = min(heavy_pairwise) if heavy_pairwise else np.nan
        self.heavy_pairwise_distance_max = max(heavy_pairwise) if heavy_pairwise else np.nan

        bond_lengths = [attributes["length"] for _, _, attributes in self.graph.edges(data=True)]
        self.bond_length_spectrum = sorted(bond_lengths)
        self.bond_length_mean = float(np.mean(bond_lengths))
        self.bond_length_std = float(np.std(bond_lengths))
        self.bond_length_min = min(bond_lengths)
        self.bond_length_max = max(bond_lengths)
        self.bond_lengths_by_type = {}
        for atom_a, atom_b, attributes in self.graph.edges(data=True):
            key = (min(int(self.atom_node_list[atom_a]), int(self.atom_node_list[atom_b])),
                   max(int(self.atom_node_list[atom_a]), int(self.atom_node_list[atom_b])),
                   attributes["bond_type"])
            self.bond_lengths_by_type.setdefault(key, []).append(attributes["length"])

        # bonded heavy-atom angles i-j-k with typed signatures
        self.heavy_angle_records = []
        for center in self.heavy_atom_indices:
            heavy_neighbors = [neighbor for neighbor in self.heavy_graph.neighbors(center)]
            for first in range(len(heavy_neighbors)):
                for second in range(first + 1, len(heavy_neighbors)):
                    atom_i, atom_k = heavy_neighbors[first], heavy_neighbors[second]
                    theta = angle_between(self.positions[atom_i], self.positions[center], self.positions[atom_k])
                    self.heavy_angle_records.append((int(self.atom_node_list[atom_i]), self.graph.edges[atom_i, center]["bond_type"],
                                                     int(self.atom_node_list[center]), self.graph.edges[center, atom_k]["bond_type"],
                                                     int(self.atom_node_list[atom_k]), theta))
        angles = [record[-1] for record in self.heavy_angle_records]
        self.heavy_angle_spectrum = sorted(angles)
        self.num_heavy_angles = len(angles)
        self.heavy_angle_mean = float(np.mean(angles)) if angles else np.nan
        self.heavy_angle_std = float(np.std(angles)) if angles else np.nan
        self.heavy_angle_min = min(angles) if angles else np.nan
        self.heavy_angle_max = max(angles) if angles else np.nan

        # heavy-atom dihedrals i-j-k-l along every heavy j-k bond; signed angle kept, cos stored alongside
        self.heavy_torsion_records = []
        rotatable_bonds = set()
        for atom_j, atom_k, attributes in self.heavy_graph.edges(data=True):
            neighbors_j = [atom for atom in self.heavy_graph.neighbors(atom_j) if atom != atom_k]
            neighbors_k = [atom for atom in self.heavy_graph.neighbors(atom_k) if atom != atom_j]
            rotatable = attributes["bond_type"] == "single" and not attributes["in_ring"] and len(neighbors_j) > 0 and len(neighbors_k) > 0
            if rotatable:
                rotatable_bonds.add((atom_j, atom_k))
            for atom_i in neighbors_j:
                for atom_l in neighbors_k:
                    phi = dihedral_angle(self.positions[atom_i], self.positions[atom_j], self.positions[atom_k], self.positions[atom_l])
                    self.heavy_torsion_records.append({
                        "atoms": (int(atom_i), int(atom_j), int(atom_k), int(atom_l)),
                        "elements": (int(self.atom_node_list[atom_i]), int(self.atom_node_list[atom_j]), int(self.atom_node_list[atom_k]), int(self.atom_node_list[atom_l])),
                        "central_bond_type": attributes["bond_type"],
                        "rotatable": rotatable,
                        "signed_degrees": phi,
                        "cos": float(np.cos(np.radians(phi))),
                    })
        self.num_heavy_torsions = len(self.heavy_torsion_records)
        self.num_rotatable_bonds = len(rotatable_bonds)
        rotatable_cos = [abs(record["cos"]) for record in self.heavy_torsion_records if record["rotatable"]]
        nonrotatable_cos = [abs(record["cos"]) for record in self.heavy_torsion_records if not record["rotatable"]]
        self.torsion_abs_cos_mean_rotatable = float(np.mean(rotatable_cos)) if rotatable_cos else np.nan
        self.torsion_abs_cos_mean_nonrotatable = float(np.mean(nonrotatable_cos)) if nonrotatable_cos else np.nan

    # ---------------- section 9: equivalence metadata ----------------
    def build_panel_equivalence(self):
        assert self.connectivity_class_id is not None, "run isomer_check before build_panel_equivalence"
        self.num_duplicates = len(self.duplicates)
        self.num_constitutional_isomers = len(self.constitutional_isomers)
        self.num_enantiomers = len(self.enantiomers)
        self.num_diastereomers = len(self.diastereomers)
        self.has_constitutional_isomer = len(self.constitutional_isomers) > 0
        self.has_enantiomer = len(self.enantiomers) > 0
        self.has_diastereomer = len(self.diastereomers) > 0

    # ---------------- section 10: QM9 targets ----------------
    def build_panel_properties(self):
        for target_name, target_value in zip(QM9_TARGET_NAMES, self.targets):
            setattr(self, target_name, float(target_value))
        assert abs(self.gap - (self.lumo - self.homo)) < 5e-3, f"{self.name}: gap column disagrees with lumo - homo"

    def build_panel_structure(self):
        """Every section that needs only this molecule; section 9 waits for My_Dataset.isomer_check"""
        self.build_panel_composition()
        self.build_panel_environments()
        self.build_panel_rings()
        self.build_panel_ring_systems()
        self.build_panel_substitution()
        self.build_panel_functional_groups()
        self.build_panel_conjugation()
        self.build_panel_graph_invariants()
        self.build_panel_geometry()
        self.build_panel_properties()

In [ ]:
class My_Dataset:
    def __init__(self, qm9_dataset):
        self.original_dataset = qm9_dataset
        self.molecules = self.create_molecules()
        self.duplicates = {}
        self.constitutional_isomers = {}
        self.enantiomers = {}
        self.diastereomers = {}
        self.stereo_undefined = {}
        self.columns = {}

    def create_molecules(self):
        molecules = []
        start = time.time()
        for index in range(len(self.original_dataset)):
            molecules.append(Molecule(self.original_dataset[index], index))
            if (index + 1) % PROGRESS_EVERY == 0:
                print(f"  built {index + 1} molecules, {time.time() - start:.0f}s elapsed")
        print(f"{len(molecules)} molecules built in {time.time() - start:.0f}s")
        return molecules

    # ---------------- isomer classification from QM9 smiles ----------------
    def isomer_check(self):
        """formula bucket -> connectivity class (flat smiles) -> stereo relation (canonical vs mirror smiles)"""
        formula_buckets = {}
        for molecule in self.molecules:
            formula_buckets.setdefault(molecule.formula, []).append(molecule)

        formula_class_id = 0
        connectivity_class_id = 0
        start = time.time()
        for bucket_number, (formula, bucket) in enumerate(formula_buckets.items()):
            graph_order = bucket[0].num_atoms
            connectivity_classes = {}
            for molecule in bucket:
                connectivity_classes.setdefault(molecule.flat_smiles, []).append(molecule)
            connectivity_classes = list(connectivity_classes.values())

            for connectivity_class in connectivity_classes:
                for molecule in connectivity_class:
                    molecule.formula_class_id = formula_class_id
                    molecule.connectivity_class_id = connectivity_class_id
                connectivity_class_id += 1
            formula_class_id += 1

            if len(connectivity_classes) > 1:
                class_index_sets = [{molecule.idx for molecule in connectivity_class} for connectivity_class in connectivity_classes]
                self.constitutional_isomers.setdefault(graph_order, []).append(class_index_sets)
                all_indices = set().union(*class_index_sets)
                for connectivity_class, index_set in zip(connectivity_classes, class_index_sets):
                    for molecule in connectivity_class:
                        molecule.constitutional_isomers.extend(sorted(all_indices - index_set))

            for connectivity_class in connectivity_classes:
                if len(connectivity_class) < 2:
                    continue
                for position_a, molecule_a in enumerate(connectivity_class):
                    duplicates = {molecule_a.idx}
                    enantiomers = {molecule_a.idx}
                    diastereomers = {molecule_a.idx}
                    stereo_undefined = {molecule_a.idx}
                    for molecule_b in connectivity_class[position_a + 1:]:
                        relation = self.classify_pair(molecule_a, molecule_b)
                        if relation == "duplicate":
                            molecule_a.duplicates.append(molecule_b.idx)
                            molecule_b.duplicates.append(molecule_a.idx)
                            duplicates.add(molecule_b.idx)
                        elif relation == "enantiomer":
                            molecule_a.enantiomers.append(molecule_b.idx)
                            molecule_b.enantiomers.append(molecule_a.idx)
                            enantiomers.add(molecule_b.idx)
                        elif relation == "diastereomer":
                            molecule_a.diastereomers.append(molecule_b.idx)
                            molecule_b.diastereomers.append(molecule_a.idx)
                            diastereomers.add(molecule_b.idx)
                        else:
                            molecule_a.stereo_undefined_pairs.append(molecule_b.idx)
                            molecule_b.stereo_undefined_pairs.append(molecule_a.idx)
                            stereo_undefined.add(molecule_b.idx)
                    if len(duplicates) > 1:
                        self.duplicates.setdefault(graph_order, []).append(duplicates)
                    if len(enantiomers) > 1:
                        self.enantiomers.setdefault(graph_order, []).append(enantiomers)
                    if len(diastereomers) > 1:
                        self.diastereomers.setdefault(graph_order, []).append(diastereomers)
                    if len(stereo_undefined) > 1:
                        self.stereo_undefined.setdefault(graph_order, []).append(stereo_undefined)
            if (bucket_number + 1) % 1000 == 0:
                print(f"  {bucket_number + 1} of {len(formula_buckets)} formula buckets, {time.time() - start:.0f}s elapsed")
        print(f"isomer_check done in {time.time() - start:.0f}s")

    def classify_pair(self, mol_a, mol_b):
        """Two molecules with the same flat smiles: duplicate, enantiomer, diastereomer, or stereo_undefined"""
        if mol_a.canonical_smiles == mol_b.canonical_smiles:
            return "duplicate"
        if mol_a.num_unassigned_stereocenters > 0 or mol_b.num_unassigned_stereocenters > 0:
            return "stereo_undefined"
        if mol_a.canonical_smiles == mol_b.mirror_smiles:
            return "enantiomer"
        return "diastereomer"

    def print_stats(self):
        """Per graph order: group counts and min/max/mean group size (size = molecules in the group)"""
        relations = {
            "duplicates": self.duplicates,
            "constitutional isomers": self.constitutional_isomers,
            "enantiomers": self.enantiomers,
            "diastereomers": self.diastereomers,
            "stereo undefined": self.stereo_undefined,
        }
        graph_orders = sorted(set().union(*[relation.keys() for relation in relations.values()]))
        print("Statistics")
        for graph_order in graph_orders:
            print(f"\n=== {graph_order} atoms ===")
            for relation_name, relation in relations.items():
                groups = relation.get(graph_order, [])
                if relation_name == "constitutional isomers":
                    group_sizes = [sum(len(connectivity_class) for connectivity_class in group) for group in groups]
                else:
                    group_sizes = [len(group) for group in groups]
                if len(groups) == 0:
                    print(f"  {relation_name}: 0 groups")
                    continue
                print(f"  {relation_name}: {len(groups)} groups, "
                      f"size min {min(group_sizes)}, max {max(group_sizes)}, mean {np.mean(group_sizes):.2f}, "
                      f"total molecules {sum(group_sizes)}")

    # ---------------- panel ----------------
    def build_panel(self):
        """Section 9 (needs isomer_check), then the column stack; the rest was built inside each Molecule"""
        for molecule in self.molecules:
            molecule.build_panel_equivalence()
        self.collect_columns()

    def collect_columns(self):
        """Stacks every panel attribute across molecules; scalars become numpy arrays, structured fields become lists"""
        for field in PANEL_SCALAR_FIELDS:
            self.columns[field] = np.array([getattr(molecule, field) for molecule in self.molecules])
        for field in PANEL_STRUCTURED_FIELDS:
            self.columns[field] = [getattr(molecule, field) for molecule in self.molecules]

    def report(self):
        """Support counts: for every scalar field, how many molecules have a nonzero value, plus range"""
        print(f"{'field':45s} {'nonzero':>8s} {'min':>10s} {'max':>10s} {'mean':>10s}")
        for field in PANEL_SCALAR_FIELDS:
            values = self.columns[field].astype(float)
            finite = values[np.isfinite(values)]
            nonzero = int((finite != 0).sum())
            print(f"{field:45s} {nonzero:8d} {finite.min():10.3f} {finite.max():10.3f} {finite.mean():10.3f}")

    def save(self):
        config = {"QM9_ROOT": QM9_ROOT, "QM9_TARGET_NAMES": QM9_TARGET_NAMES, "FUNCTIONAL_GROUP_SMARTS": FUNCTIONAL_GROUP_SMARTS,
                  "PANEL_SCALAR_FIELDS": PANEL_SCALAR_FIELDS, "PANEL_STRUCTURED_FIELDS": PANEL_STRUCTURED_FIELDS,
                  "num_molecules": len(self.molecules)}
        with open(OUTPUT_PKL, "wb") as handle:
            pickle.dump({"config": config, "columns": self.columns}, handle)
        print(f"saved {OUTPUT_PKL}")

In [ ]:
my_dataset = My_Dataset(dataset)
my_dataset.isomer_check()
my_dataset.print_stats()
my_dataset.build_panel()
my_dataset.report()
my_dataset.save()